In [1]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json

def _load_all_bboxes(bbox_dir):
    bbox_data = {}
    for bbox_file in bbox_dir.glob("*.npy"):
        frame_num = int(bbox_file.stem.split("_")[-1])
        bbox_data[frame_num] = np.load(bbox_file)
    return bbox_data

data_path = Path("/home/daniel/lab_share/scratch/omniverse_cotton_bolls/")
camera_data = {}
camera_params = {}
for row_dir in tqdm(list(data_path.iterdir())):
    if not row_dir.is_dir():
        continue
    row_num = int(row_dir.stem.split("_")[-1])
    for i in range(1, 7):
        camera_dir = row_dir / f"camera{i}_{row_num:02d}"
        # Load bounding box data.
        bbox_path = camera_dir / "bounding_box_3d"
        camera_data.setdefault(i, [])
        camera_data[i].append(_load_all_bboxes(bbox_path).copy())
        
        # Load camera parameters.
        param_path = camera_dir / Path("camera_params/camera_params_0012.json")
        camera_params[i] = json.load(param_path.open())

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [01:49<00:00, 21.88s/it]


In [2]:
import pandas as pd


def _boxes_to_world(boxes: dict) -> pd.DataFrame:
    """
    Converts loaded bounding boxes to world coordinates.
    """
    frames = []
    occlusions = []
    min_points = []
    max_points = []
    
    for frame, boxes in boxes.items():
        for box in boxes:
            world_transform = box["transform"]
            min_point = np.array([box["x_min"], box["y_min"], box["z_min"], 1])
            max_point = np.array([box["x_max"], box["y_max"], box["z_max"], 1])
            min_points.append((world_transform.T @ min_point)[:3])
            max_points.append((world_transform.T @ max_point)[:3])
            
            frames.append(frame)
            occlusions.append(box["occlusionRatio"])
            
    min_points = pd.DataFrame(min_points, columns=["x_min", "y_min", "z_min"])
    max_points = pd.DataFrame(max_points, columns=["x_max", "y_max", "z_max"])
    box_data = pd.DataFrame({"frame": frames, "occlusion": occlusions})
    box_data = pd.concat([box_data, min_points, max_points], axis=1)
    box_data.set_index("frame", inplace=True)
    return box_data


world_camera_data = {}
for camera, rows in tqdm(camera_data.items()):
    world_camera_data[camera] = []
    for row_data in rows:
        world_camera_data[camera].append(_boxes_to_world(row_data))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:24<00:00,  4.16s/it]


In [3]:
from scipy.optimize import linear_sum_assignment

def _match_bolls(frame1_bolls: pd.DataFrame, frame2_bolls: pd.DataFrame) -> tuple:
    """
    Matches bolls between two frames. Returns the number
    of matched bolls and the total number of visible bolls.
    """
    num_frame1_bolls = len(frame1_bolls)
    num_frame2_bolls = len(frame2_bolls)
    
    # Eliminate any occluded bolls.
    frame1_bolls = frame1_bolls[frame1_bolls["occlusion"] < 0.5]
    frame2_bolls = frame2_bolls[frame2_bolls["occlusion"] < 0.5]
    
    # Compute distances between every pair of bolls.
    frame1_bolls_min = frame1_bolls[["x_min", "y_min", "z_min"]].to_numpy()
    frame2_bolls_min = frame2_bolls[["x_min", "y_min", "z_min"]].to_numpy()
    min_diff = frame1_bolls_min[None, :, :] - frame2_bolls_min[:, None, :]
    min_diff = np.linalg.norm(min_diff, axis=-1)
    
    frame1_bolls_max = frame1_bolls[["x_max", "y_max", "z_max"]].to_numpy()
    frame2_bolls_max = frame2_bolls[["x_max", "y_max", "z_max"]].to_numpy()
    max_diff = frame1_bolls_max[None, :, :] - frame2_bolls_max[:, None, :]
    max_diff = np.linalg.norm(max_diff, axis=-1)
    
    total_distance = min_diff + max_diff
    
    # Use the Hungarian algorithm to find the optimal matching.
    frame1_indices, frame2_indices = linear_sum_assignment(total_distance)
    
    # Eliminate any that are too far away.
    match_costs = total_distance[frame1_indices, frame2_indices]
    num_matches = np.count_nonzero(match_costs < 0.01)
    
    return num_matches, num_frame1_bolls + num_frame2_bolls


def _match_rows(row_1: pd.DataFrame, row_2: pd.DataFrame) -> tuple:
    """
    Matches all the bolls in two rows. Returns the total numer of matched
    bolls and the total number of visible bolls.
    """
    num_matches = 0
    num_total = 0
    
    for frame in row_1.index.unique():
        row_1_frame_bolls = row_1.loc[[frame]]
        try:
            row_2_frame_bolls = row_2.loc[[frame]]
        except KeyError:
            # The second row doesn't have any bolls visible for this frame.
            num_total += len(row_1_frame_bolls)
            continue
        
        frame_matches, frame_total = _match_bolls(row_1_frame_bolls, row_2_frame_bolls)
        num_matches += frame_matches
        num_total += frame_total
        
    return num_matches, num_total

def _match_cameras(camera_1, camera_2) -> tuple:
    """
    Matches all the bolls in two cameras. Returns the total numer of matched
    bolls and the total number of visible bolls.
    """
    num_matches = 0
    num_total = 0
    
    for cam1_row, cam2_row in zip(camera_1, camera_2):
        row_matches, row_total = _match_rows(cam1_row, cam2_row)
        num_matches += row_matches
        num_total += row_total
        
    return num_matches, num_total
        

In [4]:
from itertools import combinations

# Compute matches for all camera combinations.
camera_match_ratios = {}
for cam1, cam2 in tqdm(list(combinations(range(1, 7), 2))):
    num_matches, num_total = _match_cameras(world_camera_data[cam1], world_camera_data[cam2])
    
    # Multiply by 2 so it's in range (0, 1).
    camera_match_ratios[(cam1, cam2)] = num_matches / num_total * 2

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [16:31<00:00, 66.13s/it]


In [5]:
camera_match_ratios


{
    (1, 2): 0.6195224269414368,
    (1, 3): 0.48782647967951764,
    (1, 4): 0.5765365642364325,
    (1, 5): 0.575653806429304,
    (1, 6): 0.5939565845638308,
    (2, 3): 0.586676708706655,
    (2, 4): 0.6799791068071055,
    (2, 5): 0.6269031700550325,
    (2, 6): 0.6533518621456365,
    (3, 4): 0.6084660581276261,
    (3, 5): 0.5663813336812646,
    (3, 6): 0.549585411731092,
    (4, 5): 0.6671885918028824,
    (4, 6): 0.661277261881789,
    (5, 6): 0.665827769880471
}

In [8]:
import pickle
pickle.dump(camera_match_ratios, open("camera_overlaps.pkl", "wb"))

In [ ]:
340549 / 1099392 * 2 

In [ ]:
# Compute distances between cameras.
camera_distances = {}
for cam1, cam2 in combinations(range(1, 7), 2):
    cam1_params = camera_params[cam1]
    cam2_params = camera_params[cam2]
    
    camera1_ext = np.array(cam1_params["cameraViewTransform"]).reshape((4, 4)).T
    camera2_ext = np.array(cam2_params["cameraViewTransform"]).reshape((4, 4)).T
    
    camera1_pos = (camera1_ext @ np.array([0, 0, 0, 1]))[:3]
    camera2_pos = (camera2_ext @ np.array([0, 0, 0, 1]))[:3]
    
    camera_distance = np.linalg.norm(camera1_pos - camera2_pos)
    camera_distances[(cam1, cam2)] = camera_distance

In [ ]:
camera_distances

In [ ]:
model_performance = pd.read_excel("Multi-View Model Performance.xlsx", sheet_name="MoCo 2-View Synthetic")
model_performance

In [ ]:
# Average the results.
average_performance = model_performance.groupby(["View 1", "View 2"]).mean()
average_performance.reset_index(inplace=True)

In [ ]:
import statsmodels.api as sm

# Put everything in the same order.
camera_pairs = average_performance[["View 1", "View 2"]].to_numpy() + 1
predictors = []
for cameras in camera_pairs:
    cameras = tuple(cameras)
    predictors.append([camera_match_ratios[cameras] ** 2, camera_match_ratios[cameras]])
predictors = np.array(predictors)

response = average_performance["mAP 0.5"].to_numpy()

# Add a constant for the intercept.
predictors = sm.add_constant(predictors)

model = sm.OLS(response, predictors)
results = model.fit()

results.summary()


array([[    0.66583,      50.688],
       [    0.66583,      50.688],
       [    0.66583,      50.688],
       [    0.66128,      50.947],
       [    0.66128,      50.947],
       [    0.66128,      50.947],
       [    0.66719,      3.8326],
       [    0.66719,      3.8326],
       [    0.66719,      3.8326],
       [    0.54959,      50.808],
       [    0.54959,      50.808],
       [    0.54959,      50.808],
       [    0.56638,      3.5831],
       [    0.56638,      3.5831],
       [    0.56638,      3.5831],
       [    0.60847,     0.36697],
       [    0.60847,     0.36697],
       [    0.60847,     0.36697],
       [    0.65335,      51.235],
       [    0.65335,      51.235],
       [    0.65335,      51.235],
       [     0.6269,       8.758],
       [     0.6269,       8.758],
       [     0.6269,       8.758],
       [    0.67998,      5.0772],
       [    0.67998,      5.0772],
       [    0.67998,      5.0772],
       [    0.58668,      5.2362],
       [    0.58668